# MODE 2 — FRÉGATE M2_F04 PHOTOGRAPHY — PRODUCTION

> **Pipeline complet Mode 2 From Scratch** — Caméra + Éclairage
>
> Ce notebook orchestre le pipeline de production complet :
> 1. Vérification inputs
> 2. Lancement Blender headless (camera_director.py)
> 3. Validation output
> 4. Rapport

---
**PIPELINE MODE 2** :
`LAUNCHER → M2_F01 → M2_F02 → M2_F03 → [M2_F04] → M2_F05 → M2_F06 → FINAL.mp4`

In [ ]:
#@title 🔗 [EXODUS] Drive + Session JSON
#@markdown Monte le Drive et lit exodus_session.json genere par EXO_LAUNCHER
from google.colab import drive
drive.mount('/content/drive')

import sys, json
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/EXODUS_V2"  #@param {type:"string"}
sys.path.insert(0, DRIVE_ROOT)

_session_path = Path(DRIVE_ROOT) / "exodus_session.json"
if _session_path.exists():
    with open(_session_path) as _f:
        EXODUS_SESSION = json.load(_f)
    print("OK exodus_session.json charge")
    print(f"  Mode     : {EXODUS_SESSION['mode']} --- {EXODUS_SESSION['mode_label']}")
    print(f"  Timestamp: {EXODUS_SESSION['timestamp']}")
    print(f"  Drive    : {EXODUS_SESSION['drive_root']}")
else:
    print("ATTENTION : exodus_session.json introuvable")
    print("   -> Lancer EXO_LAUNCHER.ipynb d'abord.")
    EXODUS_SESSION = {
        "mode": None, "mode_label": "UNKNOWN",
        "drive_root": DRIVE_ROOT, "status": "missing"
    }

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CONFIGURATION PRODUCTION — À REMPLIR AVANT LANCEMENT
# ═══════════════════════════════════════════════════════════════
DRIVE_ROOT   = '/content/drive/MyDrive/DRIVE_EXODUS_V2'

# Input
SCENE_BLEND  = ''   # Vide = auto-détection dans IN_SCENE_BLEND/

# Blender
BLENDER_PATH = ''   # Vide = auto-détection dans EXODUS_AI_MODELS/

# Rendu
PRESET       = 'production'   # 'production' | 'preview'
SHAKE_PRESET = 'handheld'     # 'handheld' | 'subtle' | 'aggressive'
NO_DOF       = False
NO_ATMO      = False

print('✓ Configuration production chargée')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Drive monté')

In [ ]:
# ── Pre-flight ─────────────────────────────────────────────────────────────
import os, json
from pathlib import Path

fregate = Path(DRIVE_ROOT) / '10_M2_F04_PHOTOGRAPHY'
in_scene = fregate / 'IN_SCENE_BLEND'
out_camera = fregate / 'OUT_CAMERA_READY'

blends = list(in_scene.glob('*.blend'))
print(f'IN_SCENE_BLEND/   : {len(blends)} .blend(s)')
for b in blends:
    print(f'  → {b.name}')

print(f'OUT_CAMERA_READY/ : {"existe" if out_camera.exists() else "sera créé"}')

if not blends and not SCENE_BLEND:
    raise FileNotFoundError('Aucun .blend en entrée — placez scene_ready.blend de M2_F03 dans IN_SCENE_BLEND/')

print('\n✓ Pre-flight OK — prêt pour la production')

In [ ]:
# ── Lancement production ───────────────────────────────────────────────────
import subprocess, sys

codebase = fregate / 'CODEBASE'
cmd = [
    sys.executable,
    str(codebase / 'EXO_M2_F04_PHOTOGRAPHY.py'),
    '--drive-root', DRIVE_ROOT,
    '--preset', PRESET,
    '--shake-preset', SHAKE_PRESET,
]

if SCENE_BLEND:  cmd += ['--scene', SCENE_BLEND]
if BLENDER_PATH: cmd += ['--blender-path', BLENDER_PATH]
if NO_DOF:       cmd.append('--no-dof')
if NO_ATMO:      cmd.append('--no-atmosphere')

print('=== LANCEMENT M2_F04 PHOTOGRAPHY PRODUCTION ===')
result = subprocess.run(cmd, text=True)
print(f'Code retour: {result.returncode}')

if result.returncode != 0:
    raise RuntimeError('M2_F04 échoué — vérifier logs ci-dessus')

In [ ]:
# ── Validation output + rapport ────────────────────────────────────────────
import json
from pathlib import Path

fregate = Path(DRIVE_ROOT) / '10_M2_F04_PHOTOGRAPHY'
report_path = fregate / 'OUT_REPORT' / 'm2_f04_report.json'
out_dir = fregate / 'OUT_CAMERA_READY'

if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    print(f"Statut  : {report['status']}")
    out_blend = report.get('outputs', {}).get('scene_with_camera')
    if out_blend and Path(out_blend).exists():
        size_mb = Path(out_blend).stat().st_size / 1_048_576
        print(f"Output  : {Path(out_blend).name} ({size_mb:.1f} MB)")
        print()
        print('✓ M2_F04 PHOTOGRAPHY COMPLETE')
        print('→ Transférer scene_with_camera.blend vers M2_F05 ALCHEMIST')
    else:
        print('⚠ Output .blend non trouvé')
else:
    print('Rapport non trouvé')